# 02 Data Cleaning and Preparation

Business focus: convert the raw Kaggle file into clean transaction data and SQL-ready table exports.

In [ ]:
from pathlib import Path
import pandas as pd

RAW_PATH = Path('../data/raw/global_superstore_raw.csv')
CLEAN_PATH = Path('../data/processed/global_superstore_cleaned.csv')
df = pd.read_csv(RAW_PATH, encoding='latin1')
df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('-', '_')
df.head()

In [ ]:
date_columns = ['Order_Date', 'Ship_Date']
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='%d-%m-%Y', errors='coerce')

numeric_columns = ['Sales', 'Profit', 'Discount', 'Shipping_Cost', 'Quantity']
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df.dtypes

In [ ]:
quality_checks = {
    'duplicate_row_id': df.duplicated(subset=['Row_ID']).sum() if 'Row_ID' in df.columns else None,
    'duplicate_order_id': df.duplicated(subset=['Order_ID']).sum() if 'Order_ID' in df.columns else None,
    'missing_values_total': int(df.isna().sum().sum())
}
quality_checks

In [ ]:
if {'Profit', 'Sales'}.issubset(df.columns):
    df['Profit_Margin'] = df['Profit'] / df['Sales'].replace(0, pd.NA)
if {'Order_Date', 'Ship_Date'}.issubset(df.columns):
    df['Shipping_Days'] = (df['Ship_Date'] - df['Order_Date']).dt.days
    df['Year'] = df['Order_Date'].dt.year
    df['Month'] = df['Order_Date'].dt.month
    df['Quarter'] = df['Order_Date'].dt.quarter

df.head()

In [ ]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
CLEAN_PATH